In [1]:
# For ja and es languages, the following code is to gather MHTML files from downloaded stashes of MHTML files according to banner MD5s.
LANGUAGE_TO_PROCESS = "es"

In [2]:
# gather a list of banner MD5s from the directory of GIF files for the language
import os
import shutil

# get all .gif files in the language directory
banner_md5s = []
for root, dirs, files in os.walk(LANGUAGE_TO_PROCESS):
    for file in files:
        if file.endswith(".gif"):
            # get the part of the file name before the first -
            banner_md5 = file.split("-")[0]
            banner_md5s.append(banner_md5)

print(banner_md5s)


['045853c266c0c61bc7e2072d640ef7ae', '058d43425bf474c5f3192ca3d9ef0e9b', '05a6f2b0c952e380564296aa072851f1.gif', '0ca190b4015a8cc5c86888e0ce9cc2f3', '136a3c43ebe395b0d002a27c7db23bd9', '33f220e36b5411b0635843162ff28fac', '390fa3161c26460387b7902292d8f111', '3a9962c77166d463bb39b3abb18ab03d', '3c25b0cb41deff5a42bff8a8cbf5b74b', '4198415ea02071537e0a89f9e00dbd57', '43e691071778990252329b2a6a56a1bf', '47da7384e9c5153b9c6798e7f6c43709', '4b06e57bcc131d6f53af2c521c28f9bc', '531d53df2e98da67c2123feb45f10ee4', '59ec9e7b92550320cfbe07f738433c49', '60925ca30a3462602e3e89fafc2e9fd3', '617dc548db14657365d35a79676130b7', '6c4b823f912ea07179a7603593a91813', '6f28dd2253e87c6ca2621d96cefec7a3', '70114013cae7c7fcae7562af169e976a', '7c9936fd83dda67143fe38c18f20942a', '7f6a2801cf7e0ab084a4aa843f7563e1.gif', '8b6d6f920fa85305b464c576b5c6563d', '9061606fcd3c3a8120aac44293a58957', '938fe477a1322f84016bdd66ed791bef', '968336f50f602b33d8723c8d294486fc.gif', '98633161f2fd1f7aeb8fa53b34a14a54.gif', '9b431c37cb

In [3]:

# get the first directory that starts with "mhtml" from the language directory
mhtml_dir = ""
for root, dirs, files in os.walk(LANGUAGE_TO_PROCESS):
    for dir in dirs:
        if dir.startswith("mhtml"):
            mhtml_dir = os.path.join(root, dir)
            break
    if mhtml_dir:
        break

print(mhtml_dir)

es/mhtml_intl_es


In [4]:
import email
import hashlib
import base64
import json

mhtml_results = []

# get all the MHTML files in the mhtml directory
mhtml_files = []
for root, dirs, files in os.walk(mhtml_dir):
    for file in files:
        if file.endswith(".mhtml"):
            mhtml_files.append(os.path.join(root, file))

# for each MHTML file, extract the GIFs and calculae each file's MD5 value. 
# If the MD5 value is in the list of banner MD5s, add a dictionary to the mhtml_results list with the banner MD5 and the MHTML file path.
for mhtml_file in mhtml_files:
    with open(mhtml_file, "r") as f:
        msg = email.message_from_file(f)
        for part in msg.walk():
            if part.get_content_type() == "image/gif":
                gif_data = part.get_payload(decode=True)
                gif_md5 = hashlib.md5(gif_data).hexdigest()
                if gif_md5 in banner_md5s:
                    print("Found banner MD5 {} in {}".format(gif_md5, mhtml_file))
                    mhtml_results.append({"banner_md5": gif_md5, "mhtml_file": mhtml_file})


# write the results to a JSON file
with open("mhtml_results_{}.json".format(LANGUAGE_TO_PROCESS), "w") as f:
    json.dump(mhtml_results, f, indent=4)

Found banner MD5 babe326590a58b27d8eea7bc249e1fcd in es/mhtml_intl_es/65a4721a377d4793c628cbc4/20010702065147.mhtml
Found banner MD5 e73c87497da045675054b4992c1017e0 in es/mhtml_intl_es/65a47224377d4793c628cbf0/20020205211506.mhtml
Found banner MD5 ecd8009c90f3f12e58ec60f6651d4074 in es/mhtml_intl_es/65a47228377d4793c628cbff/20000822013048.mhtml
Found banner MD5 ecd8009c90f3f12e58ec60f6651d4074 in es/mhtml_intl_es/65a47228377d4793c628cbff/20001109121000.mhtml
Found banner MD5 ecd8009c90f3f12e58ec60f6651d4074 in es/mhtml_intl_es/65a47228377d4793c628cbff/20010410204111.mhtml
Found banner MD5 ecd8009c90f3f12e58ec60f6651d4074 in es/mhtml_intl_es/65a47228377d4793c628cbff/20010606135111.mhtml
Found banner MD5 ecd8009c90f3f12e58ec60f6651d4074 in es/mhtml_intl_es/65a47228377d4793c628cbff/20010607135710.mhtml
Found banner MD5 ecd8009c90f3f12e58ec60f6651d4074 in es/mhtml_intl_es/65a47228377d4793c628cbff/20010803092356.mhtml
Found banner MD5 b9b0e4bfc347a16426f456937ce74ffb in es/mhtml_intl_es/65

In [ ]:
# read the JSON file into a dictionary
with open("mhtml_results_{}.json".format(LANGUAGE_TO_PROCESS), "r") as f:
    mhtml_results = json.load(f)

In [5]:
# consolidate mhtml_results into a dictionary with the banner MD5 as the key and the MHTML file path as the value
mhtml_results_dict = {}
for result in mhtml_results:
    mhtml_results_dict[result["banner_md5"]] = result["mhtml_file"]

# create a directory to store the MHTML files
mhtml_output_dir = "mhtml_output_{}".format(LANGUAGE_TO_PROCESS)
os.makedirs(mhtml_output_dir, exist_ok=True)

# copy the MHTML files to the output directory, and name the files banner md5 - mhtml directory name - mhtml file name
for banner_md5, mhtml_file in mhtml_results_dict.items():
    mhtml_file_name = os.path.basename(mhtml_file)
    mhtml_output_file = os.path.join(mhtml_output_dir, "{}-{}-{}".format(banner_md5, os.path.basename(os.path.dirname(mhtml_file)), mhtml_file_name))
    shutil.copy(mhtml_file, mhtml_output_file)

In [6]:
len(mhtml_results_dict)

45